In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json

train_path = '/content/drive/My Drive/SubtaskA/subtaskA_train_monolingual.jsonl'
aug_path = '/content/drive/My Drive/SubtaskA/train_monolingual_paraphrased_label0.jsonl'
save_path = '/content/drive/My Drive/SubtaskA/subtaskA_train_augmented.jsonl'

aug_map = {}
with open(aug_path, 'r', encoding='utf-8') as f:
    for line in f:
        entry = json.loads(line.strip())
        aug_map[entry['id']] = entry['text']

with open(train_path, 'r', encoding='utf-8') as fin, open(save_path, 'w', encoding='utf-8') as fout:
    for line in fin:
        entry = json.loads(line.strip())
        if entry['id'] in aug_map and entry['label'] == 0:
            entry['gen_text'] = aug_map[entry['id']]
        fout.write(json.dumps(entry, ensure_ascii=False) + '\n')

print("✅ Merged file saved to:", save_path)

✅ Merged file saved to: /content/drive/My Drive/SubtaskA/subtaskA_train_augmented.jsonl


In [ ]:
from transformers import RobertaTokenizer
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

In [ ]:
import json
import torch
from torch.utils.data import Dataset

class AIDetectorDataset(Dataset):
    def __init__(self, filepath, tokenizer, max_length=512):
        self.samples = []
        self.tokenizer = tokenizer
        self.max_length = max_length

        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                entry = json.loads(line.strip())
                self.samples.append({'text': entry['text'], 'label': int(entry['label'])})
                if entry['label'] == 0 and 'gen_text' in entry:
                    self.samples.append({'text': entry['gen_text'], 'label': 0})

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        encoding = self.tokenizer(
            item['text'],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(item['label'], dtype=torch.float)
        }

In [ ]:
train_path = '/content/drive/My Drive/SubtaskA/subtaskA_train_augmented.jsonl'
dev_path = '/content/drive/My Drive/SubtaskA/subtaskA_dev_monolingual.jsonl'

train_dataset = AIDetectorDataset(train_path, tokenizer)
dev_dataset = AIDetectorDataset(dev_path, tokenizer)

from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=32)

In [ ]:
import torch.nn as nn
from transformers import RobertaModel

class AIDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        self.classifier = nn.Sequential(
            nn.Linear(self.roberta.config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls)
        return logits

In [ ]:
import torch
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AIDetector().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
loss_fn = nn.BCEWithLogitsLoss()
epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []

    for batch in tqdm(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].unsqueeze(1).to(device)

        logits = model(input_ids, attention_mask)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = (torch.sigmoid(logits) > 0.5).int()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    print(f"[Epoch {epoch+1}] Loss: {total_loss:.4f} | Train Acc: {acc:.4f} | F1: {f1:.4f}")

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 11445/11445 [1:12:00<00:00,  2.65it/s]


[Epoch 1] Loss: 231.5750 | Train Acc: 0.9933 | F1: 0.9890


100%|██████████| 11445/11445 [1:11:57<00:00,  2.65it/s]


[Epoch 2] Loss: 83.1617 | Train Acc: 0.9977 | F1: 0.9963


100%|██████████| 11445/11445 [1:11:57<00:00,  2.65it/s]


[Epoch 3] Loss: 61.5123 | Train Acc: 0.9984 | F1: 0.9974


In [ ]:
model.eval()
val_preds, val_labels = [], []

with torch.no_grad():
    for batch in dev_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].unsqueeze(1).to(device)

        logits = model(input_ids, attention_mask)
        preds = (torch.sigmoid(logits) > 0.5).int()

        val_preds.extend(preds.cpu().numpy())
        val_labels.extend(labels.cpu().numpy())

val_acc = accuracy_score(val_labels, val_preds)
val_f1 = f1_score(val_labels, val_preds)
print(f"Validation Accuracy: {val_acc:.4f} | F1 Score: {val_f1:.4f}")

Validation Accuracy: 0.6810 | F1 Score: 0.5357
